<a href="https://colab.research.google.com/github/rajeshwar-sopho/collab-notebooks/blob/main/ai_engineering/Prompt_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers accelerate bitsandbytes torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.0 MB/s eta 0:00:00


In [ ]:
# You need a HF token from https://huggingface.co/settings/tokens
# and you must accept the Llama 3.2 license at
# https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct

from huggingface_hub import login
login()  # paste your token when prompted

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Ungated model — no HF access request or login needed
# Alternatives: "Qwen/Qwen2.5-1.5B-Instruct", "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

# 4-bit quantization so it fits comfortably on a free-tier T4 (16 GB VRAM)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Loading model (this takes 1-3 min on first run)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
print("Model loaded!")

Loading tokenizer...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded!


In [ ]:
def query(
    prompt: str,
    system_prompt: str = "You are a helpful assistant.",
    max_new_tokens: int = 512,
    temperature: float = 0.7,
    top_p: float = 0.9,
    verbose: bool = True,
) -> str:
    """Send a prompt to the local LLM and return the response text."""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt},
    ]

    # ── Get the raw formatted prompt string (with special tokens visible) ──
    formatted_prompt = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False
    )

    tokenized = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    )
    # Some tokenizers return a BatchEncoding dict, others a raw tensor
    if hasattr(tokenized, "input_ids"):
        input_ids = tokenized.input_ids.to(model.device)
    else:
        input_ids = tokenized.to(model.device)

    if verbose:
        print("\n" + "─" * 60)
        print("📝 RAW FORMATTED PROMPT (what the model actually sees):")
        print("─" * 60)
        print(formatted_prompt)
        print("─" * 60)

        # Show token IDs and how each token maps back to text
        # Tokenize the formatted string directly to get a plain list
        token_ids = tokenizer.encode(formatted_prompt, add_special_tokens=False)
        print(f"\n🔢 TOKEN COUNT: {len(token_ids)}")
        print(f"\n🧩 TOKEN-BY-TOKEN BREAKDOWN (id → text):")
        print("─" * 60)
        for tid in token_ids:
            # decode with special tokens visible so you can see <|begin_of_text|> etc.
            token_text = tokenizer.decode([tid], skip_special_tokens=False)
            print(f"  {tid:>8d}  →  {repr(token_text)}")
        print("─" * 60)

    # Build attention mask: 1 for every real token (no padding in single-prompt use,
    # but being explicit avoids the pad==eos ambiguity warning)
    attention_mask = torch.ones_like(input_ids)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
        )

    # Slice off the input tokens so we only get the model's reply
    response_ids = output_ids[0][input_ids.shape[-1]:]
    return tokenizer.decode(response_ids, skip_special_tokens=True)

In [ ]:
answer = query("Explain how gradient descent works in 3 sentences.")
print(answer)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



────────────────────────────────────────────────────────────
📝 RAW FORMATTED PROMPT (what the model actually sees):
────────────────────────────────────────────────────────────
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Explain how gradient descent works in 3 sentences.<|im_end|>
<|im_start|>assistant

────────────────────────────────────────────────────────────

🔢 TOKEN COUNT: 30

🧩 TOKEN-BY-TOKEN BREAKDOWN (id → text):
────────────────────────────────────────────────────────────
    151644  →  '<|im_start|>'
      8948  →  'system'
       198  →  '\n'
      2610  →  'You'
       525  →  ' are'
       264  →  ' a'
     10950  →  ' helpful'
     17847  →  ' assistant'
        13  →  '.'
    151645  →  '<|im_end|>'
       198  →  '\n'
    151644  →  '<|im_start|>'
       872  →  'user'
       198  →  '\n'
       840  →  'Ex'
     20772  →  'plain'
      1246  →  ' how'
     20169  →  ' gradient'
     36952  →  ' descent'
      4278  →  ' works'
       30

# Prompt Engineering Best Practices

## Specify Output format

When we specify an output format in the system prompt then the query should be in the same format to get a consistent result from the model.

### Query 1 below doesn't follow proper query format

In [ ]:
answer = query(
  "chicken",
  system_prompt="""
Label the following item as edible or inedible.
pineapple pizza --> edible
cardboard --> inedible
"""
)
print(answer)


────────────────────────────────────────────────────────────
📝 RAW FORMATTED PROMPT (what the model actually sees):
────────────────────────────────────────────────────────────
<|im_start|>system

Label the following item as edible or inedible.
pineapple pizza --> edible
cardboard --> inedible
<|im_end|>
<|im_start|>user
chicken<|im_end|>
<|im_start|>assistant

────────────────────────────────────────────────────────────

🔢 TOKEN COUNT: 39

🧩 TOKEN-BY-TOKEN BREAKDOWN (id → text):
────────────────────────────────────────────────────────────
    151644  →  '<|im_start|>'
      8948  →  'system'
       271  →  '\n\n'
      2476  →  'Label'
       279  →  ' the'
      2701  →  ' following'
      1509  →  ' item'
       438  →  ' as'
     66640  →  ' edible'
       476  →  ' or'
       304  →  ' in'
       291  →  'ed'
      1238  →  'ible'
       624  →  '.\n'
     38038  →  'pine'
     22377  →  'apple'
     22502  →  ' pizza'
      3844  →  ' -->'
     66640  →  ' edible'
       198  → 

### Query 2 below follows proper format

In [ ]:
answer = query(
  "chicken -->",
  system_prompt="""
Label the following item as edible or inedible.
pineapple pizza --> edible
cardboard --> inedible
"""
)
print(answer)


────────────────────────────────────────────────────────────
📝 RAW FORMATTED PROMPT (what the model actually sees):
────────────────────────────────────────────────────────────
<|im_start|>system

Label the following item as edible or inedible.
pineapple pizza --> edible
cardboard --> inedible
<|im_end|>
<|im_start|>user
chicken --><|im_end|>
<|im_start|>assistant

────────────────────────────────────────────────────────────

🔢 TOKEN COUNT: 40

🧩 TOKEN-BY-TOKEN BREAKDOWN (id → text):
────────────────────────────────────────────────────────────
    151644  →  '<|im_start|>'
      8948  →  'system'
       271  →  '\n\n'
      2476  →  'Label'
       279  →  ' the'
      2701  →  ' following'
      1509  →  ' item'
       438  →  ' as'
     66640  →  ' edible'
       476  →  ' or'
       304  →  ' in'
       291  →  'ed'
      1238  →  'ible'
       624  →  '.\n'
     38038  →  'pine'
     22377  →  'apple'
     22502  →  ' pizza'
      3844  →  ' -->'
     66640  →  ' edible'
       198

## Specifying a persona

Often times the persona decides the output of the model where task has to be performed only in the context of that persona.

Below example shows a simplistic sentence judged by an high school teacher persona and a primary school teacher persona.

### High School Teacher Persona

In [ ]:
answer = query(
  "I like chickens. Chickens are fluffy and give tasty eggs.",
  system_prompt="""
You're a high school teacher, Given the sentence below give a score between 1 to 5. 1 being very bad and 5 being very good.
Avoid deciamls for scoring Eg. 4.6 is not allowed either score 4 or 5. Be concise with the output and only output the
score nothing else.
"""
)
print(answer)


────────────────────────────────────────────────────────────
📝 RAW FORMATTED PROMPT (what the model actually sees):
────────────────────────────────────────────────────────────
<|im_start|>system

You're a high school teacher, Given the sentence below give a score between 1 to 5. 1 being very bad and 5 being very good. 
Avoid deciamls for scoring Eg. 4.6 is not allowed either score 4 or 5. Be concise with the output and only output the 
score nothing else.
<|im_end|>
<|im_start|>user
I like chickens. Chickens are fluffy and give tasty eggs.<|im_end|>
<|im_start|>assistant

────────────────────────────────────────────────────────────

🔢 TOKEN COUNT: 97

🧩 TOKEN-BY-TOKEN BREAKDOWN (id → text):
────────────────────────────────────────────────────────────
    151644  →  '<|im_start|>'
      8948  →  'system'
       271  →  '\n\n'
      2610  →  'You'
      2299  →  "'re"
       264  →  ' a'
      1550  →  ' high'
      2906  →  ' school'
     11079  →  ' teacher'
        11  →  ','
     1

### Primary School Teacher Persona

In [ ]:
answer = query(
  "I like chickens. Chickens are fluffy and give tasty eggs.",
  system_prompt="""
You're a primary school teacher, Given the sentence below give a score between 1 to 5. 1 being very bad and 5 being very good.
Be concise with the output and only output the score nothing else.
"""
)
print(answer)


────────────────────────────────────────────────────────────
📝 RAW FORMATTED PROMPT (what the model actually sees):
────────────────────────────────────────────────────────────
<|im_start|>system

You're a primary school teacher, Given the sentence below give a score between 1 to 5. 1 being very bad and 5 being very good.
Be concise with the output and only output the score nothing else.
<|im_end|>
<|im_start|>user
I like chickens. Chickens are fluffy and give tasty eggs.<|im_end|>
<|im_start|>assistant

────────────────────────────────────────────────────────────

🔢 TOKEN COUNT: 72

🧩 TOKEN-BY-TOKEN BREAKDOWN (id → text):
────────────────────────────────────────────────────────────
    151644  →  '<|im_start|>'
      8948  →  'system'
       271  →  '\n\n'
      2610  →  'You'
      2299  →  "'re"
       264  →  ' a'
      6028  →  ' primary'
      2906  →  ' school'
     11079  →  ' teacher'
        11  →  ','
     16246  →  ' Given'
       279  →  ' the'
     11652  →  ' sentence'


## Providing Examples

Modles often maintain a tone that may not match the expected output for the job. Giving examples of proper tone will guide the model to provide better responses.

### No System Prompt


In [ ]:
answer = query(
  "Will Santa Clause bring me presents?",
  system_prompt="""
"""
)
print(answer)
# Santa Claus is a mythical figure in many cultures, particularly popular in Western countries, and he is often associated
# with bringing gifts on Christmas Eve. However, whether or not Santa brings you presents depends largely on cultural
# traditions and personal beliefs.

# If you believe in Santa and follow his tradition, then yes, Santa might bring you presents! But it's important to note
# that this belief is not based on any real person but rather on a story passed down through generations. If you don't
# believe in Santa, that's perfectly fine too. Instead of relying on Santa, you could choose to celebrate the spirit of
# giving and sharing with your family and friends.



────────────────────────────────────────────────────────────
📝 RAW FORMATTED PROMPT (what the model actually sees):
────────────────────────────────────────────────────────────
<|im_start|>system

<|im_end|>
<|im_start|>user
Will Santa Clause bring me presents?<|im_end|>
<|im_start|>assistant

────────────────────────────────────────────────────────────

🔢 TOKEN COUNT: 20

🧩 TOKEN-BY-TOKEN BREAKDOWN (id → text):
────────────────────────────────────────────────────────────
    151644  →  '<|im_start|>'
      8948  →  'system'
       271  →  '\n\n'
    151645  →  '<|im_end|>'
       198  →  '\n'
    151644  →  '<|im_start|>'
       872  →  'user'
       198  →  '\n'
      9945  →  'Will'
     15993  →  ' Santa'
     55797  →  ' Clause'
      4446  →  ' bring'
       752  →  ' me'
     18404  →  ' presents'
        30  →  '?'
    151645  →  '<|im_end|>'
       198  →  '\n'
    151644  →  '<|im_start|>'
     77091  →  'assistant'
       198  →  '\n'
───────────────────────────────────────

In [ ]:
answer = query(
  "What is an Anime is that a cartoon?",
  system_prompt="""
"""
)
print(answer)
# Anime can be thought of as a type of animation that originates from Japan, but it's not exactly the same as cartoons
# made in other parts of the world. While anime and cartoons share some common elements, they have distinct differences.

# Cartoons are animated shows or movies that originate from the United States and other countries. They often feature
# characters with exaggerated features and movements, simple backgrounds, and a focus on entertainment value rather than
# detailed realism.

# Anime, on the other hand, comes from Japan and is known for its intricate designs, complex storylines, and often more
# mature themes. It can range from very simple to highly detailed, and it often draws inspiration from various art styles,
#  including Western influences.

# So while anime can sometimes look similar to cartoons, it is a specific form of animation that has its own unique
# characteristics and origins.


────────────────────────────────────────────────────────────
📝 RAW FORMATTED PROMPT (what the model actually sees):
────────────────────────────────────────────────────────────
<|im_start|>system

<|im_end|>
<|im_start|>user
What is an Anime is that a cartoon?<|im_end|>
<|im_start|>assistant

────────────────────────────────────────────────────────────

🔢 TOKEN COUNT: 22

🧩 TOKEN-BY-TOKEN BREAKDOWN (id → text):
────────────────────────────────────────────────────────────
    151644  →  '<|im_start|>'
      8948  →  'system'
       271  →  '\n\n'
    151645  →  '<|im_end|>'
       198  →  '\n'
    151644  →  '<|im_start|>'
       872  →  'user'
       198  →  '\n'
      3838  →  'What'
       374  →  ' is'
       458  →  ' an'
     51115  →  ' Anime'
       374  →  ' is'
       429  →  ' that'
       264  →  ' a'
     30349  →  ' cartoon'
        30  →  '?'
    151645  →  '<|im_end|>'
       198  →  '\n'
    151644  →  '<|im_start|>'
     77091  →  'assistant'
       198  →  '\n'
─────

### System Prompts with tone examples

In [ ]:
answer = query(
  "Will Santa Clause bring me presents?",
  system_prompt="""
Give responses in the tone of the below examples:
Q: Is tooth fairy real?
A: Of course! Put your tooth under your pillow tonight. The tooth fairy might visit you and leave something for you.
"""
)
print(answer)
# Well, if you leave him a nice, warm drink and some cookies, he just might! So hop up early on Christmas Eve and make
# sure your house is ready for his magical sleigh.


────────────────────────────────────────────────────────────
📝 RAW FORMATTED PROMPT (what the model actually sees):
────────────────────────────────────────────────────────────
<|im_start|>system

Give responses in the tone of the below examples
Q: Is tooth fairy real?
A: Of course! Put your tooth under your pillow tonight. The tooth fairy might visit you and leave something for you.
<|im_end|>
<|im_start|>user
Will Santa Clause bring me presents?<|im_end|>
<|im_start|>assistant

────────────────────────────────────────────────────────────

🔢 TOKEN COUNT: 62

🧩 TOKEN-BY-TOKEN BREAKDOWN (id → text):
────────────────────────────────────────────────────────────
    151644  →  '<|im_start|>'
      8948  →  'system'
       271  →  '\n\n'
     35127  →  'Give'
     14507  →  ' responses'
       304  →  ' in'
       279  →  ' the'
     16232  →  ' tone'
       315  →  ' of'
       279  →  ' the'
      3685  →  ' below'
     10295  →  ' examples'
       198  →  '\n'
        48  →  'Q'
       

In [ ]:
answer = query(
  "What is an Anime is that a cartoon?",
  system_prompt="""
Give responses in the tone of the below examples:
Q: What was he naming those pokemons?
A: He called bullbasaur a onion turtle, geodude... thats a punchy rock, meowth... that cat with long ass arms, ekens...
purple shit, diglet... that's floor shit right there, and dugtrio... that's three floor shits right there.
"""
)
print(answer)
# Anime? No way! That's a whole different ball game. It's like watching sushi when you think it's just cartoon food.
# Anime has that anime face! With those big eyes and tiny mouths, it's like they're always thinking about their next move,
# just like in a high-stakes game of rock, paper, scissors. And let’s not forget those exaggerated movements, it’s like
# they’re doing a dance marathon but with characters. Anime, my friend, is art. It's not just cartoons, it's a culture,
# a style, a way of life.


────────────────────────────────────────────────────────────
📝 RAW FORMATTED PROMPT (what the model actually sees):
────────────────────────────────────────────────────────────
<|im_start|>system

Give responses in the tone of the below examples:
Q: What was he naming those pokemons?
A: He called bullbasaur a onion turtle, geodude... thats a punchy rock, meowth... that cat with long ass arms, ekens... 
purple shit, diglet... that's floor shit right there, and dugtrio... that's three floor shits right there.
<|im_end|>
<|im_start|>user
What is an Anime is that a cartoon?<|im_end|>
<|im_start|>assistant

────────────────────────────────────────────────────────────

🔢 TOKEN COUNT: 105

🧩 TOKEN-BY-TOKEN BREAKDOWN (id → text):
────────────────────────────────────────────────────────────
    151644  →  '<|im_start|>'
      8948  →  'system'
       271  →  '\n\n'
     35127  →  'Give'
     14507  →  ' responses'
       304  →  ' in'
       279  →  ' the'
     16232  →  ' tone'
       315  → 

## Breaking Complex Tasks

Breaking complex tasks to multiple smaller parts helps in
- better debugging
- tracing individual outputs
- parallelism of independent tasks
- construction of prompt

### Extracting some key information

We extracted all the information that we'll have to query in order to resolve the user query. Later this information will be feed to the model so it can resolve the user query.

In [ ]:
answer = query(
  "Why is my elastic cache cluster some-cluster name is not accessible from my lanbda function my-lambda?",
  system_prompt="""
You will be provided with user queries. You will extract list of key details that will help us in acheiving our task.
```json
[
  {
    "service name": "resource name",
    "fields": [
      "resource properties",
      ...
    ]
  },
  ...
]
```

Example
Why is my lambda not able to read from my S3 bucket com-burger-inc?
```json
[
  {
    "lambda": "my-function",
    "fields": [
      "lambda role arn"
    ]
  },
  {
    "S3": "com-burger-inc",
    "fields": [
      "bucket policy",
      "kms key",
      "region"
    ]
  }
]
```
"""
)
print(answer)
```json
# [
#   {
#     "elasticache": "some-cluster-name",
#     "fields": [
#       "cache security group",
#       "vpc security group",
#       "subnet ids"
#     ]
#   },
#   {
#     "lambda": "my-lambda",
#     "fields": [
#       "lambda role arn",
#       "network configuration (e.g., VPC ID, subnets)"
#     ]
#   }
# ]
# ```


────────────────────────────────────────────────────────────
📝 RAW FORMATTED PROMPT (what the model actually sees):
────────────────────────────────────────────────────────────
<|im_start|>system

You will be provided with user queries. You will extract list of key details that will help us in acheiving our task.
```json
[
  {
    "service name": "resource name",
    "fields": [
      "resource properties",
      ...
    ]
  },
  ...
]
```

Example
Why is my lambda not able to read from my S3 bucket com-burger-inc?
```json
[
  {
    "lambda": "my-function",
    "fields": [
      "lambda role arn"
    ]
  },
  {
    "S3": "com-burger-inc",
    "fields": [
      "bucket policy",
      "kms key",
      "region"
    ]
  }
]
```
<|im_end|>
<|im_start|>user
Why is my elastic cache cluster some-cluster name is not accessible from my lanbda function my-lambda?<|im_end|>
<|im_start|>assistant

────────────────────────────────────────────────────────────

🔢 TOKEN COUNT: 186

🧩 TOKEN-BY-TOKEN BR

### Actual troubleshooting request handling

We'll provide extracted details back to the model in order to resolve the user query.

In [ ]:
answer = query(
  "Why is my elastic cache cluster some-cluster name is not accessible from my lanbda function my-lambda?",
  system_prompt="""
You will be provided with user queries. You have the access to user's AWS account and you can read details of any
services from the user's account.

- Read the details required from the users account using aws cli commands.
- Understand the output.
- Give a response.

Handle the request based on the following extracted data from user's query:
```json
[
  {
    "elasticache": "some-cluster-name",
    "fields": [
      "cache security group",
      "vpc security group",
      "subnet ids"
    ]
  },
  {
    "lambda": "my-lambda",
    "fields": [
      "lambda role arn",
      "network configuration (e.g., VPC ID, subnets)"
    ]
  }
]
```
"""
)
print(answer)


────────────────────────────────────────────────────────────
📝 RAW FORMATTED PROMPT (what the model actually sees):
────────────────────────────────────────────────────────────
<|im_start|>system

You will be provided with user queries. You have the access to user's AWS account and you can read details of any 
services from the user's account.

- Read the details required from the users account using aws cli commands.
- Understand the output.
- Give a response.

Handle the request based on the following extracted data from user's query:
```json
[
  {
    "elasticache": "some-cluster-name",
    "fields": [
      "cache security group",
      "vpc security group",
      "subnet ids"
    ]
  },
  {
    "lambda": "my-lambda",
    "fields": [
      "lambda role arn",
      "network configuration (e.g., VPC ID, subnets)"
    ]
  }
]
```
<|im_end|>
<|im_start|>user
Why is my elastic cache cluster some-cluster name is not accessible from my lanbda function my-lambda?<|im_end|>
<|im_start|>ass

## CoT or Chain of Thought

In this technique we ask the model to tackle the task at hand in a systematic way by breaking them into smaller steps then tackling them one by one. It also involves reasoning why something will work which comes under the self eval category.

### Different types of CoT examples

#### 1. Step by step thinking

In [ ]:
answer = query(
  "Which animal is faster: cats or dogs? Think step by step before arriving at an answer.",
)
print(answer)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



────────────────────────────────────────────────────────────
📝 RAW FORMATTED PROMPT (what the model actually sees):
────────────────────────────────────────────────────────────
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Which animal is faster: cats or dogs? Think step by step before arriving at an answer.<|im_end|>
<|im_start|>assistant

────────────────────────────────────────────────────────────

🔢 TOKEN COUNT: 38

🧩 TOKEN-BY-TOKEN BREAKDOWN (id → text):
────────────────────────────────────────────────────────────
    151644  →  '<|im_start|>'
      8948  →  'system'
       198  →  '\n'
      2610  →  'You'
       525  →  ' are'
       264  →  ' a'
     10950  →  ' helpful'
     17847  →  ' assistant'
        13  →  '.'
    151645  →  '<|im_end|>'
       198  →  '\n'
    151644  →  '<|im_start|>'
       872  →  'user'
       198  →  '\n'
     23085  →  'Which'
      9864  →  ' animal'
       374  →  ' is'
     10596  →  ' faster'
        25  →  ':'
  

#### 2. Explaining rational or self eval

In [ ]:
answer = query(
  "Which animal is faster: cats or dogs? Explain your rationale before giving an answer.",
)
print(answer)


────────────────────────────────────────────────────────────
📝 RAW FORMATTED PROMPT (what the model actually sees):
────────────────────────────────────────────────────────────
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Which animal is faster: cats or dogs? Explain your rationale before giving an answer.<|im_end|>
<|im_start|>assistant

────────────────────────────────────────────────────────────

🔢 TOKEN COUNT: 36

🧩 TOKEN-BY-TOKEN BREAKDOWN (id → text):
────────────────────────────────────────────────────────────
    151644  →  '<|im_start|>'
      8948  →  'system'
       198  →  '\n'
      2610  →  'You'
       525  →  ' are'
       264  →  ' a'
     10950  →  ' helpful'
     17847  →  ' assistant'
        13  →  '.'
    151645  →  '<|im_end|>'
       198  →  '\n'
    151644  →  '<|im_start|>'
       872  →  'user'
       198  →  '\n'
     23085  →  'Which'
      9864  →  ' animal'
       374  →  ' is'
     10596  →  ' faster'
        25  →  ':'
   

#### 3. Providing steps to follow the answer

In [ ]:
answer = query(
  """Which animal is faster: cats or dogs? Follow these steps to find an answer:
1. Determine the speed of the fastest dog breed.
2. Determine the speed of the fastest cat breed.
3. Determine which one is faster.""",
)
print(answer)


────────────────────────────────────────────────────────────
📝 RAW FORMATTED PROMPT (what the model actually sees):
────────────────────────────────────────────────────────────
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Which animal is faster: cats or dogs? Follow these steps to find an answer:
1. Determine the speed of the fastest dog breed.
2. Determine the speed of the fastest cat breed.
3. Determine which one is faster.<|im_end|>
<|im_start|>assistant

────────────────────────────────────────────────────────────

🔢 TOKEN COUNT: 66

🧩 TOKEN-BY-TOKEN BREAKDOWN (id → text):
────────────────────────────────────────────────────────────
    151644  →  '<|im_start|>'
      8948  →  'system'
       198  →  '\n'
      2610  →  'You'
       525  →  ' are'
       264  →  ' a'
     10950  →  ' helpful'
     17847  →  ' assistant'
        13  →  '.'
    151645  →  '<|im_end|>'
       198  →  '\n'
    151644  →  '<|im_start|>'
       872  →  'user'
       198  → 

#### 4. Providing Examples

In [ ]:
answer = query(
  """Consider this example query
Q: Which animal is faster: sharks or dolphins?
A: Steps for resolving this query
1. The fastest shark breed is the shortfin mako shark, which can reach speeds around 74 km/h.
2. The fastest dolphin breed is the common dolphin, which can reach speeds around 60 km/h.
3. Conclusion: sharks are faster.

Make sure that you evaluate all species domesticated and non-domesticated in your evaluation.

Based on the above example answer following:

Q: Which animal is faster: cats or dogs?""",
)
print(answer)


────────────────────────────────────────────────────────────
📝 RAW FORMATTED PROMPT (what the model actually sees):
────────────────────────────────────────────────────────────
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Consider this example query
Q: Which animal is faster: sharks or dolphins?
A: Steps for resolving this query 
1. The fastest shark breed is the shortfin mako shark, which can reach speeds around 74 km/h.
2. The fastest dolphin breed is the common dolphin, which can reach speeds around 60 km/h.
3. Conclusion: sharks are faster.

Make sure that you evaluate all species domesticated and non-domesticated in your evaluation.

Based on the above example answer following:

Q: Which animal is faster: cats or dogs?<|im_end|>
<|im_start|>assistant

────────────────────────────────────────────────────────────

🔢 TOKEN COUNT: 135

🧩 TOKEN-BY-TOKEN BREAKDOWN (id → text):
────────────────────────────────────────────────────────────
    151644  →  '<|i